In [ ]:
import sys
import sktime
import tqdm as tq
import xgboost as xgb
import matplotlib
import seaborn as sns
import sklearn as skl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sktime.forecasting.model_selection import temporal_train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sktime.utils.plotting import plot_series
from xgboost import XGBRegressor

In [ ]:
wh_2020 = pd.read_csv('2020.csv', encoding = 'cp949')
wh_2021 = pd.read_csv('2021.csv', encoding = 'cp949')
wh_2022 = pd.read_csv('2022.csv', encoding = 'cp949')

aws_2020= pd.read_csv('2020aws.csv', encoding = 'CP949')
aws_2021= pd.read_csv('2021aws.csv', encoding = 'CP949')
aws_2022= pd.read_csv('2022aws.csv', encoding = 'CP949')

In [ ]:
date = wh_2020.loc[:,'관측 일시'].to_list()
a = pd.DataFrame({'Date':date[0]
                  ,'잠수교':wh_2020.iloc[0,1:].T})

for i in range(1,wh_2020.shape[0]):
    b = pd.DataFrame({'Date':date[i],
                     '잠수교':wh_2020.iloc[i,1:].T})
    a = pd.concat([a,b])

a.reset_index(inplace=True)

a['일시'] = a['Date'].map(str)+' '+ a['index'].map(str)

for i in range(0,a.shape[0]):
    a.loc[i,'일시'] = a.loc[i,'일시'].replace('시',':00:00')
    
for i in range(0,a.shape[0]):
    a.loc[i,'일시'] = a.loc[i,'일시'].replace('24:00:00','00:00:00')
a.sort_values('일시',inplace=True)
a.drop(['index','Date'], axis=1, inplace=True)
a.reset_index(drop=True, inplace=True)
a['일시'] = pd.to_datetime(a['일시'])
a = a.astype({'일시':'str'})
wh_2020 = a.copy()

In [ ]:
date = wh_2021.loc[:,'관측 일시'].to_list()
a = pd.DataFrame({'Date':date[0]
                  ,'잠수교':wh_2021.iloc[0,1:].T})

for i in range(1,wh_2021.shape[0]):
    b = pd.DataFrame({'Date':date[i],
                     '잠수교':wh_2021.iloc[i,1:].T})
    a = pd.concat([a,b])

a.reset_index(inplace=True)

a['일시'] = a['Date'].map(str)+' '+ a['index'].map(str)

for i in range(0,a.shape[0]):
    a.loc[i,'일시'] = a.loc[i,'일시'].replace('시',':00:00')
    
for i in range(0,a.shape[0]):
    a.loc[i,'일시'] = a.loc[i,'일시'].replace('24:00:00','00:00:00')
a.sort_values('일시',inplace=True)
a.drop(['index','Date'], axis=1, inplace=True)
a.reset_index(drop=True, inplace=True)
a['일시'] = pd.to_datetime(a['일시'])
a = a.astype({'일시':'str'})
wh_2021 = a.copy()

In [ ]:
date = wh_2022.loc[:,'관측 일시'].to_list()
a = pd.DataFrame({'Date':date[0]
                  ,'잠수교':wh_2022.iloc[0,1:].T})

for i in range(1,wh_2022.shape[0]):
    b = pd.DataFrame({'Date':date[i],
                     '잠수교':wh_2022.iloc[i,1:].T})
    a = pd.concat([a,b])

a.reset_index(inplace=True)

a['일시'] = a['Date'].map(str)+' '+ a['index'].map(str)

for i in range(0,a.shape[0]):
    a.loc[i,'일시'] = a.loc[i,'일시'].replace('시',':00:00')
    
for i in range(0,a.shape[0]):
    a.loc[i,'일시'] = a.loc[i,'일시'].replace('24:00:00','00:00:00')
a.sort_values('일시',inplace=True)
a.drop(['index','Date'], axis=1, inplace=True)
a.reset_index(drop=True, inplace=True)
a['일시'] = pd.to_datetime(a['일시'])
a = a.astype({'일시':'str'})
wh_2022 = a.copy()

In [ ]:
aws_2020.drop(['지점','일사(MJ/m^2)','일조(hr)'], axis=1, inplace = True)
aws_2020['일시'] = pd.to_datetime(aws_2020['일시'])

aws_2020_filtered = aws_2020[aws_2020['일시'].between('2020-05-15', '2020-10-15 23:00:00')]
aws_2020_filtered = aws_2020_filtered.astype({'일시':'str'})

mixed_2020 = pd.merge(wh_2020, aws_2020_filtered, on='일시', how='outer',indicator=False)
mixed_2020.iloc[:,2:9]=mixed_2020.iloc[:,2:9].interpolate(method='linear')
mixed_2020.columns = ['target','date','temp','winddir','windpow','rain','localpre','seapre','humid']
mixed_2020 = mixed_2020[['date','temp','winddir','windpow','rain','localpre','seapre','humid','target']]

In [ ]:
aws_2021.drop(['지점','일사(MJ/m^2)','일조(hr)'], axis=1, inplace = True)
aws_2021['일시'] = pd.to_datetime(aws_2021['일시'])

aws_2021_filtered = aws_2021[aws_2021['일시'].between('2021-05-15', '2021-10-15 23:00:00')]
aws_2021_filtered = aws_2021_filtered.astype({'일시':'str'})

mixed_2021 = pd.merge(wh_2021, aws_2021_filtered, on='일시', how='outer',indicator=False)
mixed_2021.iloc[:,2:9]=mixed_2021.iloc[:,2:9].interpolate(method='linear')
mixed_2021.columns = ['target','date','temp','winddir','windpow','rain','localpre','seapre','humid']
mixed_2021 = mixed_2021[['date','temp','winddir','windpow','rain','localpre','seapre','humid','target']]

In [ ]:
aws_2022.drop(['지점','일사(MJ/m^2)','일조(hr)'], axis=1, inplace = True)
aws_2022['일시'] = pd.to_datetime(aws_2022['일시'])

aws_2022_filtered = aws_2022[aws_2022['일시'].between('2022-05-15', '2022-10-15 23:00:00')]
aws_2022_filtered = aws_2022_filtered.astype({'일시':'str'})

mixed_2022 = pd.merge(wh_2022, aws_2022_filtered, on='일시', how='outer',indicator=False)
mixed_2022.iloc[:,2:9]=mixed_2022.iloc[:,2:9].interpolate(method='linear')
mixed_2022.columns = ['target','date','temp','winddir','windpow','rain','localpre','seapre','humid']
mixed_2022 = mixed_2022[['date','temp','winddir','windpow','rain','localpre','seapre','humid','target']]

In [ ]:
train = pd.concat([mixed_2020,mixed_2021,mixed_2022], axis=0)

In [ ]:
train

In [ ]:
train['target_shift'] = train['target'].shift(2).rolling(window=2).mean()

In [ ]:
train

In [ ]:
train.iloc[0:3,9]=train.iloc[0:3,8].mean()

In [ ]:
train

In [ ]:
#train['target'] = train['target']-float(train['target'].median())

In [ ]:
#scaler = MinMaxScaler()
#train_minmax = scaler.fit_transform(train.iloc[:,1:9])
#train_minmax_df = pd.DataFrame(train_minmax, columns=train.columns[1:9])
#train.iloc[:,1:9]= train_minmax_df
#train

In [ ]:
train.reset_index(drop=True, inplace=True)

In [ ]:
train.iloc[9440:9460]

In [ ]:
date = pd.to_datetime(train['date'])

train['hour'] = date.dt.hour
train['day'] = date.dt.weekday
train['month'] = date.dt.month
train['week'] = date.dt.isocalendar().week
train['sin_time']= np.sin(2*np.pi*train.hour/24)
train['cos_time'] = np.cos(2*np.pi*train.hour/24)

train.drop('hour', axis=1, inplace=True)
train.drop('date', axis=1, inplace=True)
train['week'] = train['week'].astype(int)
train['target'] = train['target'].astype(float)
y = train['target_shift'].astype(float)
x = train.drop(['target_shift'], axis=1)

In [ ]:
train.iloc[4050:4065]

### Temporal train-test

In [ ]:
y_train, y_test, x_train, x_test = temporal_train_test_split(y=y, X=x, test_size=3696)
y_train_1, y_valid, x_train_1, x_valid = temporal_train_test_split(y=y_train, X=x_train, test_size=3696)

In [ ]:
x_train

In [ ]:
xgb = XGBRegressor()
xgb.fit(x_train_1, y_train_1)

In [ ]:
pred = xgb.predict(x_test)
pred = pd.Series(pred)

In [ ]:
mae = mean_absolute_error(y_test, pred)
mae

### train-test

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size=1/3, random_state=123)
X_train_1, X_valid, Y_train_1, Y_valid = train_test_split(X_train, Y_train, test_size=1/2, random_state=123)

In [ ]:
import hyperopt
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

In [ ]:
# regularization candiate 정의
reg_candidate = [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1, 5, 10, 100]

# space 정의, Hyperparameter의 이름을 key 값으로 입력
space={'max_depth': hp.quniform("max_depth", 1, 32, 1),
       'learning_rate': hp.quniform ('learning_rate', 0.01, 0.05, 0.005),
       'reg_alpha' : hp.choice('reg_alpha', reg_candidate),
       'reg_lambda' : hp.choice('reg_lambda', reg_candidate),
       'subsample': hp.quniform('subsample', 0.6, 1, 0.05),
       'colsample_bytree' : hp.quniform('colsample_bytree', 0.6, 1, 0.05),
       'min_child_weight' : hp.quniform('min_child_weight', 1, 10, 1),
       'n_estimators': hp.quniform('n_estimators', 200, 1500, 100)
      }

# 목적 함수 정의
# n_estimators, max_depth와 같은 반드시 int 타입을 가져야 하는 hyperparamter는 int로 타입 캐스팅 합니다.
def hyperparameter_tuning(space):
    model=XGBRegressor(n_estimators =int(space['n_estimators']), 
                       max_depth = int(space['max_depth']), 
                       learning_rate = space['learning_rate'],
                       reg_alpha = space['reg_alpha'],
                       reg_lambda = space['reg_lambda'],
                       subsample = space['subsample'],
                       colsample_bytree = space['colsample_bytree'], 
                       min_child_weight = int(space['min_child_weight']),
                       random_state=1024, 
                      )
    
    evaluation = [(x_train_1, y_train_1), (x_valid, y_valid)]
    
    model.fit(x_train_1, y_train_1,
              eval_set=evaluation, 
              eval_metric="mape",
              early_stopping_rounds=20,
              verbose=0)

    pred = model.predict(x_valid)
    mape= mean_absolute_percentage_error(y_valid, pred)    
    # 평가 방식 선정
    return {'loss':mape, 'status': STATUS_OK, 'model': model}

In [ ]:
# Trials 객체 선언합니다.
trials = Trials()
# best에 최적의 하이퍼 파라미터를 return 받습니다.
best = fmin(fn=hyperparameter_tuning,
            space=space,
            algo=tpe.suggest,
            max_evals=100, # 최대 반복 횟수를 지정합니다.
            trials=trials)

# 최적화된 결과를 int로 변환해야하는 파라미터는 타입 변환을 수행합니다.
best['max_depth'] = int(best['max_depth'])
best['min_child_weight'] = int(best['min_child_weight'])
best['n_estimators'] = int(best['n_estimators'])
best['reg_alpha'] = reg_candidate[int(best['reg_alpha'])]
best['reg_lambda'] = reg_candidate[int(best['reg_lambda'])]
best['random_state'] = 1024
print (best)

In [ ]:
xgb = XGBRegressor(**best)
xgb.fit(x_train, y_train)
pred = xgb.predict(x_test)
pred = pd.Series(pred)
mape = mean_absolute_percentage_error(y_test, pred)
mape

In [ ]:
import numpy as np

def smape(true, pred):
    v = 2 * abs(pred - true) / (abs(pred) + abs(true))
    output = np.mean(v) * 100
    return output

In [ ]:
smape(y_test, pred)

In [ ]:
plt.figure(figsize=(24,8))
plt.plot(y_test.sort_index().index,y_test,'b')
plt.plot(y_test.sort_index().index, pred,'r')

In [ ]:
xgb = XGBRegressor()
xgb.fit(X_train_1, Y_train_1)

In [ ]:
pred = xgb.predict(X_test)
pred = pd.Series(pred)

In [ ]:
mae = mean_absolute_error(Y_test, pred)
mae

In [ ]:
import shap
shap.initjs()

In [ ]:
explainer = shap.Explainer(xgb)
shap_values = explainer(x_test)

In [ ]:
shap.plots.waterfall(shap_values[0])

In [ ]:
shap.plots.force(shap_values[0])